In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "duguid2020strategies")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Duguid_2020_duguidetal_2021_pureco_evapecognition.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df['study_id']="duguid2020strategies"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df.columns


In [3]:
df['condition_temp']=df['condition']
df['condition_temp'].replace('solo', np.nan, inplace=True)

# df['dyad_temp_1'] = df['pair_name'].str.slice(0,3)
# df['dyad_temp_2'] = df['pair_name'].str.slice(3,6)


In [4]:
dyad_1=[]
dyad_2=[]
for index, row in df.iterrows():
    if not pd.isna(row['condition_temp']):
        dyad_1.append(row['ind_left'])
    else:
        dyad_1.append("")
df = df.assign(dyad_1=dyad_1)
for index, row in df.iterrows(): 
    if not pd.isna(row['condition_temp']):
        dyad_2.append(row['ind_right'])
    else:
        dyad_2.append("")
df = df.assign(dyad_2=dyad_2)


In [5]:
df.rename(columns={"name": "ape",
    "sess_part": "session_part",
    "prepost": "pre_post",
    "partner_no": "partner_number",
    "day":"day_original",
    "date":"day",
    "session":"session_original"}, inplace=True)

In [6]:
df['session_original'] = df['session_original'].astype(str)
df['session_part'] = df['session_part'].astype(str)
df['session_part'].replace('nan', '', inplace=True, regex=True)

In [7]:
df['session']=df.session_original.str.cat(df.session_part, sep='')

In [8]:
# ape_left = []
# ape_right = []
# for index, row in df.iterrows():
#     if row['condition'] == 'solo':
#         ape_left.append("")
#         ape_right.append("")
#     else:
#         ape_left.append(row['ape_left_temp'])
#         ape_right.append(row['ape_right_temp'])
# df = df.assign(ape_left=ape_left)
# df = df.assign(ape_right=ape_right)

In [9]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
# df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)
    df['dyad_1'].replace(x, y, inplace=True)
    df['dyad_2'].replace(x, y, inplace=True)
    # df['ape_left'].replace(x, y, inplace=True)
    # df['ape_right'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left')


In [10]:
left_df = df.loc[~df['side'].str.contains('r')]
left_df = left_df.assign(role='individual_left')
left_df.rename(columns={"trial_code":"trial_code_temp",
    "button1": "button_1_left",
    "button2":"button_2_left",
    "button3":"button_3_left",
    "button4":"button_4_left",
    "success":"success_left",
    "room":"room_left",
    "food":"food_left",
    "pre_post":"pre_post_left",
    "achieve_coord":"achieve_coord_left",
    "num_choices":"num_choices_left",
    "com_coded":"com_coded_left",
    "timing_choice":"timing_choice_left",
    "comm_type":"comm_type_left",
    "comm_mod":"comm_mod_left",
    "comm_timing":"comm_timing_left"}, inplace=True)


In [11]:
right_df = df.loc[~df['side'].str.contains('l')]
right_df = right_df.assign(role_2='individual_right')
right_df.rename(columns={"ape": "ape_2_temp",
    "button1": "button_1_right",
    "button2":"button_2_right",
    "button3":"button_3_right",
    "button4":"button_4_right",
    "success":"success_right",
    "room":"room_right",
    "food":"food_right",
    "pre_post":"pre_post_right",
    "achieve_coord":"achieve_coord_right",
    "num_choices":"num_choices_right",
    "com_coded":"com_coded_right",
    "timing_choice":"timing_choice_right",
    "comm_type":"comm_type_right",
    "comm_mod":"comm_mod_right",
    "comm_timing":"comm_timing_right"}, inplace=True)

In [12]:
df = left_df.merge(right_df, left_on='trial_code_temp', right_on='trial_code', suffixes=('', '_y'))
# fulldf = fulldf.loc[:,~fulldf.columns.duplicated()]

In [13]:
ape_temp=[]
for index, row in df.iterrows():
    if row['condition'] == 'solo':
        ape_temp.append(np.nan)
    else:
        ape_temp.append(row['ape_2_temp'])
df = df.assign(ape_2=ape_temp)
df = df.assign(role='subject')

In [14]:
comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
df= df.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

In [15]:
df['ape'] = df['ape'].str.rstrip()
df['ape_2'] = df['ape_2'].str.rstrip()

In [16]:
dyad=[]
for index, row in df.iterrows():
    if not pd.isna(row['ape_2']):
        dyad.append(row['ape']+'_'+row['ape_2'])
    else:
        dyad.append("")
df = df.assign(dyad=dyad)
df = df.assign(role='subject')

In [17]:
role=[]
for index, row in df.iterrows():
    if row['condition'] == 'solo':
        role.append("subject")
    else:
        role.append("individual_left")
df = df.assign(role=role)

In [18]:
df.rename(columns={"ape": "participant","ape_2":"participant_2", 
                   "achieve_coord_left":"achieve_coord",
                   "group":"species_subgroup"}, inplace=True)
# df.columns

In [19]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
df= df.merge(ape_dob,left_on='participant', right_on='name', how='left')

comp_path_birth_dates_2 = os.path.join(pathway_gen, "apes_age_calculations_2.csv")
ape_dob_2 = pd.read_csv(comp_path_birth_dates_2) 
df= df.merge(ape_dob_2,left_on='participant_2', right_on='name_2', how='left')
two_participant_lists = [['dodc','dob','age_in_years'],
                        ['dodc_2','dob_2','age_in_years_2']]
for x,y,k in two_participant_lists:
    df[x] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
    df[x] = pd.to_datetime(df[x])
    df[y] = pd.to_datetime(df[y])
    df[k] = (df[x] - df[y]).dt.days//365

In [20]:
duguid2020strategies_standardized=df[['study_id','year','month','day',  
                                      'participant', 'age_in_years','sex', 'role',
                                      'participant_2','age_in_years_2','sex_2' ,'role_2','species','species_subgroup', 'dyad',
      'session', 'trial',
      'condition','trial_code','partner_number','achieve_coord',
       'room_left','pre_post_left',    
        'food_left',  'button_1_left', 'button_2_left',
       'button_3_left', 'button_4_left', 'success_left', 
       'num_choices_left', 'com_coded_left', 'timing_choice_left',
       'comm_type_left', 'comm_mod_left', 'comm_timing_left',  
         'room_right', 'pre_post_right', 'food_right', 'button_1_right', 'button_2_right', 'button_3_right',
       'button_4_right', 'success_right', 
       'num_choices_right', 'com_coded_right', 'timing_choice_right',
       'comm_type_right', 'comm_mod_right', 'comm_timing_right']].copy()


In [21]:
duguid2020strategies_standardized['role'].replace('subject', 'focal_participant', inplace=True, regex=True)
duguid2020strategies_standardized['participant_2'].replace(np.nan, '', inplace=True)
duguid2020strategies_standardized.loc[duguid2020strategies_standardized.participant_2 == '', ['role_2']] = ''

# duguid2020strategies_standardized=duguid2020strategies_standardized.sort_values(by = ['participant','session','trial'])

In [22]:
comp_out_path_stand = os.path.join(out_pathway, 'duguid2020strategies_exp1_standardized.csv')
duguid2020strategies_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

names =duguid2020strategies_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
duguid2020strategies_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'duguid2020strategies_exp1_glossary.csv')
duguid2020strategies_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

